In [1]:
import os
import time
import calendar

import requests
import pandas as pd
from dotenv import load_dotenv

In [2]:
load_dotenv("../api_keys.env", override=True)

TOKEN = os.getenv("TMDB_ACCESS_TOKEN")

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "accept": "application/json"
}

discover_url = "https://api.themoviedb.org/3/discover/movie"

In [3]:
def get_movie_details(movie_id):
    url = f"https://api.themoviedb.org/3/movie/{movie_id}"

    try:
        response = requests.get(url, headers=headers, timeout=10)

        if response.status_code == 200:
            return response.json()

        print("Request failed:", movie_id, response.status_code)
        return None

    except requests.exceptions.Timeout:
        print("Request timed out:", movie_id)
        return None

In [4]:
def is_blockbuster(movie, minimum_budget=100_000_000):
    return movie["budget"] >= minimum_budget

In [5]:
def create_movie_record(movie):
    genres = [genre["name"] for genre in movie["genres"]]
    studios = [company["name"] for company in movie["production_companies"]]

    return {
        "movie_id": movie["id"],
        "title": movie["title"],
        "release_date": movie["release_date"],
        "studio": ", ".join(studios),
        "genre": ", ".join(genres),
        "production_budget": movie["budget"],
        "worldwide_gross": movie["revenue"],
        "runtime": movie["runtime"],
        "audience_score": movie["vote_average"],
        "vote_count": movie["vote_count"],
        "popularity": movie["popularity"]
    }

In [6]:
def collect_movies(movie_ids):
    movies_data = []

    # enumerate() gives both the count number and the movie ID.
    for index, movie_id in enumerate(movie_ids, start=1):
        print(f"Checking {index} of {len(movie_ids)}: {movie_id}")

        movie = get_movie_details(movie_id)

        if movie is not None and is_blockbuster(movie):
            movie_record = create_movie_record(movie)
            movies_data.append(movie_record)
            print("Added:", movie["title"])

    return movies_data

In [7]:
def get_movie_ids_for_month(year, month):
    last_day = calendar.monthrange(year, month)[1]

    start_date = f"{year}-{month:02d}-01"
    end_date = f"{year}-{month:02d}-{last_day:02d}"

    first_page_params = {
        "primary_release_date.gte": start_date,
        "primary_release_date.lte": end_date,
        "with_release_type": "2|3",
        "sort_by": "primary_release_date.asc",
        "page": 1
    }

    try:
        response = requests.get(
            discover_url,
            headers=headers,
            params=first_page_params,
            timeout=10
        )
    except requests.exceptions.Timeout:
        print(f"Timed out for {year}-{month:02d}, page 1")
        return []

    if response.status_code != 200:
        print(f"Request failed for {year}-{month:02d}, page 1:", response.status_code)
        return []

    data = response.json()

    if "results" not in data:
        print(f"No results found for {year}-{month:02d}, page 1")
        print(data)
        return []

    # .get() avoids a crash if total_pages is missing from the API response.
    total_pages = data.get("total_pages", 0)

    movie_ids = [
        movie["id"]
        for movie in data["results"]
    ]

    for page in range(2, total_pages + 1):
        page_params = first_page_params.copy()
        page_params["page"] = page

        # Small pause helps avoid hitting the API too aggressively.
        time.sleep(0.25)

        try:
            response = requests.get(
                discover_url,
                headers=headers,
                params=page_params,
                timeout=10
            )
        except requests.exceptions.Timeout:
            print(f"Timed out for {year}-{month:02d}, page {page}")
            continue

        if response.status_code != 200:
            print(f"Request failed for {year}-{month:02d}, page {page}:", response.status_code)
            continue

        data = response.json()

        if "results" not in data:
            print(f"No results found for {year}-{month:02d}, page {page}")
            print(data)
            continue

        page_movie_ids = [
            movie["id"]
            for movie in data["results"]
        ]

        movie_ids.extend(page_movie_ids)

    print(f"{year}-{month:02d}: {len(movie_ids)} candidate IDs")

    return movie_ids

In [8]:
def get_movie_ids_for_year(year):
    all_year_ids = []

    for month in range(1, 13):
        monthly_ids = get_movie_ids_for_month(year, month)
        all_year_ids.extend(monthly_ids)

    # set() removes duplicate movie IDs before converting back to a list.
    unique_year_ids = list(set(all_year_ids))

    print(f"{year}: {len(unique_year_ids)} unique candidate IDs")

    return unique_year_ids

In [9]:
def movies_to_dataframe(movies_data):
    movies_df = pd.DataFrame(movies_data)

    movies_df["release_date"] = pd.to_datetime(
        movies_df["release_date"],
        errors="coerce"
    )

    movies_df["release_year"] = movies_df["release_date"].dt.year
    movies_df["release_month"] = movies_df["release_date"].dt.month

    column_order = [
        "movie_id",
        "title",
        "release_date",
        "release_year",
        "release_month",
        "studio",
        "genre",
        "production_budget",
        "worldwide_gross",
        "runtime",
        "audience_score",
        "vote_count",
        "popularity"
    ]

    movies_df = movies_df[column_order]

    return movies_df

In [10]:
def collect_blockbusters_for_year(year):
    year_ids = get_movie_ids_for_year(year)
    year_blockbusters = collect_movies(year_ids)
    year_df = movies_to_dataframe(year_blockbusters)

    output_path = f"../data/raw/tmdb_blockbusters_{year}.csv"

    year_df.to_csv(output_path, index=False)

    print(f"Saved {year} CSV:", output_path)
    print(f"{year} blockbusters:", len(year_df))

    return year_df

In [11]:
def collect_blockbusters_for_year_range(start_year, end_year):
    all_years_df = []

    # end_year + 1 is needed because range() stops before the final number.
    for year in range(start_year, end_year + 1):
        print(f"\nStarting {year}")

        year_df = collect_blockbusters_for_year(year)
        all_years_df.append(year_df)

    # pd.concat stacks the yearly DataFrames into one combined table.
    combined_df = pd.concat(all_years_df, ignore_index=True)

    output_path = f"../data/raw/tmdb_blockbusters_{start_year}_{end_year}.csv"

    combined_df.to_csv(output_path, index=False)

    print("Saved combined CSV:", output_path)
    print("Combined rows:", len(combined_df))

    return combined_df

In [ ]:
blockbusters_2015_2025_df = collect_blockbusters_for_year_range(2015, 2025)
#Run this to save tables


Starting 2015


In [14]:
#Make a cleaned starter file

In [15]:
# cleaned_starter_df = blockbusters_2015_2025_df.copy()

# cleaned_starter_df["ip_type"] = ""
# cleaned_starter_df["franchise"] = ""

# column_order = [
#     "movie_id",
#     "title",
#     "release_date",
#     "release_year",
#     "release_month",
#     "studio",
#     "genre",
#     "ip_type",
#     "franchise",
#     "production_budget",
#     "worldwide_gross",
#     "runtime",
#     "audience_score",
#     "vote_count",
#     "popularity"
# ]

# cleaned_starter_df = cleaned_starter_df[column_order]

# cleaned_starter_df.to_csv(
#     "../data/cleaned/blockbuster_movies_cleaned_starter.csv",
#     index=False
# )